# Hidden Sardinian Gems — attrattività OSM

Questo notebook è rerunnable end-to-end. Combina gli indicatori turistici esportati dai notebook, l'indice di attrattività calcolato da POI OpenStreetMap e il bonus separato per grotte e spiagge pure. Se i CSV OSM sono già presenti li carica offline; in caso contrario avvia il fetch Overpass con le definizioni di `fetch_osm_data.py`.


In [ ]:
import json, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def display(value):
    print(value)

ROOT = Path('/workspace')
TMP = ROOT / 'tmp'
OUT = ROOT / 'data' / 'sardegna-gems'
YEARS = [2022, 2023, 2024, 2025]
PILLARS = ['turismo', 'natura', 'ristorazione', 'servizi', 'infrastrutture']
WEIGHTS = {'turismo': .30, 'natura': .25, 'ristorazione': .20, 'servizi': .15, 'infrastrutture': .10}
print('Ambiente pronto:', OUT)


## 1–4. Indicatori turistici e geometria

Gli indicatori di pressione restano quelli del progetto: densità turistica, intensità turistica, densità ricettiva e utilizzazione lorda. La geometria comunale è il layer locale dei 377 comuni sardi.


In [ ]:
overtourism = {y: pd.read_csv(TMP / f'indice_overtourism_{y}.csv', encoding='utf-8-sig') for y in YEARS}
intensita = {y: pd.read_csv(TMP / f'intensita_turistica_{y}.csv', encoding='utf-8-sig') for y in YEARS}
ricettiva = {y: pd.read_csv(TMP / f'densita_ricettiva_{y}.csv', encoding='utf-8-sig') for y in YEARS}
densita = {y: pd.read_csv(TMP / f'densita_turistica_{y}.csv', encoding='utf-8-sig') for y in YEARS}
print('CSV turistici caricati:', len(overtourism[2025]), 'righe nel 2025')
with open(OUT / 'sardinia_municipalities.geojson', encoding='utf-8') as f:
    geo = json.load(f)
assert len(geo['features']) == 377
print('Geometria:', len(geo['features']), 'comuni')


## Sezione 4b: Indice di Attrattività da OpenStreetMap

I POI OSM sono classificati in cinque pilastri: **turismo (30%)**, **natura (25%)**, **ristorazione (20%)**, **servizi (15%)**, **infrastrutture (10%)**. Il conteggio è associato ai comuni con `geopandas.sjoin(predicate='within')`; ogni pilastro è densità per km², normalizzata min-max a 0–100. L'indice è la somma ponderata. Il dataset OSM pre-computato è la modalità offline predefinita.


In [ ]:
osm_counts_path = OUT / 'osm_per_comune.csv'
osm_index_path = OUT / 'indice_attrattivita.csv'
if not osm_counts_path.exists():
    subprocess.check_call([sys.executable, str(OUT / 'fetch_osm_data.py')])
if not osm_index_path.exists():
    subprocess.check_call([sys.executable, str(OUT / 'compute_attractiveness.py')])
osm_counts = pd.read_csv(osm_counts_path, dtype={'codice_istat': str})
osm_index = pd.read_csv(osm_index_path, dtype={'codice_istat': str})
assert len(osm_counts) == len(osm_index) == 377
assert set(PILLARS).issubset(osm_counts.columns)
print('OSM CSV:', len(osm_counts), 'comuni; indice:', osm_index.indice.min(), '–', osm_index.indice.max())


In [ ]:
# Controllo del calcolo: area UTM 32N, densità, min-max e somma ponderata.
import geopandas as gpd
boundaries = gpd.read_file(OUT / 'sardinia_municipalities.geojson')
boundaries['codice_istat'] = boundaries['com_istat_code'].astype(str)
boundaries['nome_comune'] = boundaries['name'].astype(str)
boundaries = boundaries.to_crs('EPSG:32632')
boundaries['area_km2'] = boundaries.geometry.area / 1_000_000
check = boundaries[['codice_istat','nome_comune','area_km2']].merge(osm_counts, on=['codice_istat','nome_comune'], how='left')
for p in PILLARS:
    density = check[p] / check.area_km2
    check[p] = (density - density.min()) / (density.max() - density.min()) * 100
check['indice'] = sum(check[p] * WEIGHTS[p] for p in PILLARS)
# Esecuzione esplicita dello spatial join sui punti cache quando disponibili.
from shapely.geometry import Point
from importlib.util import spec_from_file_location, module_from_spec
spec = spec_from_file_location('fetch_osm_data', OUT / 'fetch_osm_data.py')
osm_mod = module_from_spec(spec); spec.loader.exec_module(osm_mod)
if any(path.exists() for path in osm_mod.CACHE_CANDIDATES):
    cached_points = osm_mod.load_cache()
    cached_gdf = gpd.GeoDataFrame(cached_points, geometry=[Point(p['lon'], p['lat']) for p in cached_points], crs='EPSG:4326').to_crs('EPSG:32632')
    joined = gpd.sjoin(cached_gdf, boundaries[['codice_istat','nome_comune','geometry']], how='inner', predicate='within')
    print('Spatial join punti cache:', len(joined), 'associazioni; conteggi precomputati:', len(osm_counts))
else:
    print('Spatial join punti: rinviato; CSV OSM pre-computato disponibile offline')
print('Spatial join CRS:', boundaries.crs, '| indice ricalcolato:', round(check.indice.min(), 2), '–', round(check.indice.max(), 2))
top_osm = check.nlargest(10, 'indice')[['nome_comune','indice']]
display(top_osm)
top_osm.sort_values('indice').plot.barh(x='nome_comune', y='indice', figsize=(8,4), legend=False, title='Top 10 comuni — attrattività OSM')
plt.tight_layout()


## 5. Grotte, spiagge pure e bonus marino

Il bonus marino è separato dall'attrattività OSM: è mantenuto dal database curato già presente nel `data.json` e applicato solo come moltiplicatore `(1 + marine_gem_bonus)`.


In [ ]:
# Il builder conserva il database marino e ricalcola tutte le mensilità con l'indice OSM.
subprocess.check_call([sys.executable, str(OUT / 'build_data_with_osm.py')])
with open(OUT / 'data.json', encoding='utf-8') as f:
    payload = json.load(f)
assert len(payload['comuni']) == 377
assert all('attractiveness_index' in c and len(c['attractiveness_pillars']) == 5 for c in payload['comuni'])
print('Payload rigenerato:', len(payload['comuni']), 'comuni')


## 6. Algoritmo Hidden Gem

`ATTRACTIVENESS = osm_index / 100` sostituisce completamente il vecchio quality multiplier (costa, dimensione, infrastrutture). La stagionalità, l'eligibilità (`n_indicatori >= 2`, presenze > 0, overtourism < p75) e il bonus marino restano invariati.

`GEM_SCORE = (1 - overtourism) × ATTRACTIVENESS × (1 + seasonal_adjustment) × (1 + marine_gem_bonus)`


In [ ]:
# Ranking mensile dal JSON appena rigenerato; la formula è quella sopra.
comuni = pd.DataFrame([{
    'comune': c['name'], 'attractiveness_index': c['attractiveness_index'],
    'marine_gem_bonus': c['marine_gem_bonus'], 'pillars': c['attractiveness_pillars']
} for c in payload['comuni']])
rows = []
for c in payload['comuni']:
    r = next(x for x in c['monthly'] if x['year'] == 2025 and x['month'] == 8)
    if r['eligible']:
        attract = c['attractiveness_index'] / 100
        score = (1-r['overtourism']) * attract * (1+r['seasonal_adjustment']) * (1+c['marine_gem_bonus'])
        rows.append({'comune': c['name'], 'score': score, 'osm_index': c['attractiveness_index'], 'overtourism': r['overtourism']})
scores = pd.DataFrame(rows).sort_values(['score','comune'], ascending=[False,True])
scores['score_0_100'] = scores.score / scores.score.max() * 100
display(scores.head(10))


## 7–8. Risultati e visualizzazioni

La classifica e la figura usano il punteggio con attrattività OSM; il breakdown dei cinque pilastri è disponibile in `attractiveness_pillars` per ogni comune.


In [ ]:
top = scores.head(15).sort_values('score_0_100')
fig, ax = plt.subplots(figsize=(8,5))
ax.barh(top.comune, top.score_0_100, color='#287c8e')
ax.set_xlabel('Gem score normalizzato 0–100')
ax.set_title('Hidden Sardinian Gems — agosto 2025')
plt.tight_layout()


## 9. Rigenerazione e verifica della mappa HTML

`build_data_with_osm.py` aggiorna `data.json`; il passaggio seguente sostituisce il JSON embedded nell'HTML mantenendo una mappa autosufficiente.


In [ ]:
html_path = OUT / 'index.html'
html = html_path.read_text(encoding='utf-8')
open_tag = '<script type="application/json" id="embedded-data">'
close_tag = '</script>'
start = html.find(open_tag)
end = html.find(close_tag, start + len(open_tag))
assert start >= 0 and end >= 0
embedded = json.dumps(payload, ensure_ascii=False, separators=(',', ':')).replace('</', '<\/')
html_path.write_text(html[:start+len(open_tag)] + embedded + html[end:], encoding='utf-8')
check_html = html_path.read_text(encoding='utf-8')
assert 'function calcScore' in check_html and 'attractiveness_index' in check_html
assert 'quality_multiplier' not in check_html
print('PASS — notebook outputs verified:', len(payload['comuni']), 'comuni; HTML embedded JSON aggiornato')
